In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=4e3c856283a4c37a613f0614711d6ae72e53ea882a89f3222b594232646aa475
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import time

In [3]:
import torch
from datasets import load_from_disk
from transformers import  (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, TrainingArguments, Trainer, set_seed
)
import json

In [ ]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

In [15]:
dataset_path = '/content/drive/MyDrive/Colab Notebooks/dl/proj_fashion_ner/fashion_dataset_dict'

model_name = "deepvk/RuModernBERT-small"

MAX_LENGTH = 128
SEED = 1
EPOCHS = 3
BSZ = 8
LR = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/dl/proj_fashion_ner/'

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Загрузка данных

In [9]:

ds = load_from_disk(dataset_path)

with open(f"{dataset_path}/label_maps.json", "r", encoding="utf-8") as f:
    maps = json.load(f)

label2id = maps["label2id"]
id2label = {int(k): v for k,v in maps["id2label"].items()}
tag_weight = maps["tag_weight"]

#Токенизация данных

In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

выравнивание токенов относительно BIO-тэгов из-за разбиения слов на сабворды токенизатором

In [ ]:
# ПРОВЕРКА ДАННЫХ

from collections import Counter

# имя столбца с метками
LABEL_COL = "tags"  # или "ner_tags" если у тебя так называется

def check_lengths(ds, split="train"):
    bad = []
    for i, (toks, labs) in enumerate(zip(ds[split]["tokens"], ds[split][LABEL_COL])):
        if len(toks) != len(labs):
            bad.append((i, len(toks), len(labs)))
    return bad

bad_train = check_lengths(ds, "train")
bad_val = check_lengths(ds, "validation") if "validation" in ds else []
bad_test = check_lengths(ds, "test") if "test" in ds else []

print("bad train:", len(bad_train), bad_train[:5])
print("bad val:", len(bad_val), bad_val[:5])
print("bad test:", len(bad_test), bad_test[:5])

# ПРОВЕРКА ДАННЫХ

bad train: 0 []
bad val: 0 []
bad test: 0 []


In [ ]:
label_all_tokens = False

def tokenize_and_align(batch):
    tokenized = tokenizer(batch["tokens"], is_split_into_words=True,
                          truncation=True, padding=True)
    aligned_labels = []
    for i, word_ids in enumerate(tokenized.word_ids(batch_index=j) for j in range(len(batch["tokens"]))):
        word_labels = batch["tags"][i]
        new_labels = []
        prev_word_id = None
        for wid in word_ids:
            if wid is None:
                new_labels.append(-100)
            elif wid != prev_word_id:
                new_labels.append(word_labels[wid])
            else:
                new_labels.append(word_labels[wid] if label_all_tokens else -100)
            prev_word_id = wid
        aligned_labels.append(new_labels)

    tokenized["labels"] = aligned_labels
    return tokenized

In [ ]:
cols = ["tokens", "tags"]
ds_tok = ds.map(tokenize_and_align, batched=True, remove_columns=cols)

Map:   0%|          | 0/668 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/167 [00:00<?, ? examples/s]

#Загрузка модели

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/138M [00:00<?, ?B/s]

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at deepvk/RuModernBERT-small and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# Загрузка предобученной модели

from safetensors.torch import load_file

model_path = '/content/drive/MyDrive/Colab Notebooks/dl/proj_fashion_ner/checkpoint-252/model.safetensors'
model_state_dict = load_file(model_path)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)
model.load_state_dict(model_state_dict)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/138M [00:00<?, ?B/s]

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at deepvk/RuModernBERT-small and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<All keys matched successfully>

In [17]:
model.to(device)

ModernBertForTokenClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 384, padding_idx=50283)
      (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=384, out_features=1152, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=384, out_features=384, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=384, out_features=1152, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=576, out_features=384, bias=False)
        )
      )
  

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

Для расчета метрик вспомогательные функции:

In [ ]:
def align_preds_and_labels(preds, labels):
    preds = np.argmax(preds, axis=2)
    y_true, y_pred = [], []
    for true_seq, pred_seq in zip(labels, preds):
        true_tags, pred_tags = [], []
        for t, p in zip(true_seq, pred_seq):
            if t != -100:
                true_tags.append(id2label[int(t)])
                pred_tags.append(id2label[int(p)])
        y_true.append(true_tags)
        y_pred.append(pred_tags)
    return y_true, y_pred

def compute_metrics(p):
    preds, labels = p
    y_true, y_pred = align_preds_and_labels(preds, labels)
    return {
        "precision": precision_score(y_true, y_pred),
        "recall":    recall_score(y_true, y_pred),
        "f1":        f1_score(y_true, y_pred),
    }

#Обучение модели

In [ ]:
args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BSZ,
    per_device_eval_batch_size=BSZ,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    metric_for_best_model="f1",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipython-input-2774515742.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
W0902 18:53:56.856000 1944 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.727200,0.105850,0.962594,0.914692,0.938032
2,0.089900,0.108141,0.954436,0.943128,0.948749
3,0.033000,0.111273,0.947991,0.950237,0.949112


TrainOutput(global_step=252, training_loss=0.20412949416490775, metrics={'train_runtime': 92.7391, 'train_samples_per_second': 21.609, 'train_steps_per_second': 2.717, 'total_flos': 24855182085888.0, 'train_loss': 0.20412949416490775, 'epoch': 3.0})

#Оценка модели

In [ ]:
pred = trainer.predict(ds_tok["test"])
y_true, y_pred = align_preds_and_labels(pred.predictions, pred.label_ids)

print(f"Eval P/R/F1: {precision_score(y_true, y_pred):.3f} / {recall_score(y_true, y_pred):.3f} / {f1_score(y_true, y_pred):.3f}\n")
print(classification_report(y_true, y_pred, digits=3))

Eval P/R/F1: 0.948 / 0.950 / 0.949

              precision    recall  f1-score   support

       EVENT      0.906     0.923     0.914        52
        FEAT      0.959     0.947     0.953       225
        ITEM      0.946     0.966     0.956       145

   micro avg      0.948     0.950     0.949       422
   macro avg      0.937     0.945     0.941       422
weighted avg      0.948     0.950     0.949       422



#Проверка инфиренса

In [11]:
def predict_tokens(tokens):
    enc = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    enc.to(device)
    model.eval()

    time_start = time.time()
    with torch.no_grad():
        out = model(**enc)
        pred_ids = out.logits.argmax(-1).squeeze().tolist()
        word_ids = enc.word_ids(0)
    result = []
    seen = set()
    for idx, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        result.append((tokens[wid], id2label[pred_ids[idx]]))

    time_stop = time.time()
    time_delta = time_stop - time_start
    print(f'total time: {time_delta//60:.0f} min {time_delta % 60:.4f} sec')
    return result

In [28]:
question_for_inference = 'Здравствуйте. Немогли бы вы отправить мне ссылку на синие прошлогодние джинсы'
predict_tokens(question_for_inference.split())

total time: 0 min 0.0559 sec


[('Здравствуйте.', 'O'),
 ('Немогли', 'O'),
 ('бы', 'O'),
 ('вы', 'O'),
 ('отправить', 'O'),
 ('мне', 'O'),
 ('ссылку', 'O'),
 ('на', 'O'),
 ('синие', 'B-FEAT'),
 ('прошлогодние', 'B-FEAT'),
 ('джинсы', 'B-ITEM')]

In [23]:
question_for_inference = 'на прошлой неделе купил тулуп и сапоги!))  что можете посоветовать одеть на рыбалку'
predict_tokens(question_for_inference.split())

total time: 0 min 0.0227 sec


[('на', 'O'),
 ('прошлой', 'B-FEAT'),
 ('неделе', 'O'),
 ('купил', 'O'),
 ('тулуп', 'O'),
 ('и', 'O'),
 ('сапоги!))', 'B-ITEM'),
 ('что', 'O'),
 ('можете', 'O'),
 ('посоветовать', 'O'),
 ('одеть', 'O'),
 ('на', 'O'),
 ('рыбалку', 'B-EVENT')]